# 02 实验分组与随机化检查

检验是否有明显异常，但不能从检查通过直接推断随机化一定成立。公开分配日志不可用。

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root))
import pandas as pd
from src.data import download
from src.config import OUT
df = download()
print(f"客户数：{len(df):,}；字段数：{len(df.columns)}")

客户数：64,000；字段数：12


## 一、三组样本量与 SRM
1:1:1 是本项目的计划比例假设。SRM 用卡方拟合优度检验，告警阈值 0.001。

In [2]:
from src.statistics import balance
validation = balance(df)
print(df.segment.value_counts().to_string())
print(validation)

segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
{'planned_ratio_assumption': '1:1:1; allocation logs unavailable', 'srm_chi2': 0.20253125000000002, 'srm_p': 0.9036929575604196, 'srm_flag': False, 'max_abs_smd': 0.016899120365944938, 'threshold_smd': 0.1, 'warning': 'Balance and SRM checks cannot prove random assignment.'}


## 二、实验前平衡
SMD 衡量组间变量分布差异。分类变量逐类别转为指示变量，0.1 是检查参考线。

In [3]:
smd = pd.read_csv(OUT / 'balance_smd.csv')
print(smd.assign(abs_smd=smd.smd.abs()).sort_values('abs_smd', ascending=False).head(10).to_string(index=False))

    treatment       control         feature            level   mean_t   mean_c       smd  abs_smd
  Mens E-Mail Womens E-Mail         channel            Phone 0.433660 0.442044 -0.016899 0.016899
  Mens E-Mail     No E-Mail history_segment   2) $100 - $200 0.220162 0.226978 -0.016360 0.016360
  Mens E-Mail Womens E-Mail         channel              Web 0.445394 0.437368  0.016162 0.016162
  Mens E-Mail     No E-Mail history_segment      7) $1,000 + 0.021777 0.019525  0.015835 0.015835
  Mens E-Mail Womens E-Mail history_segment 6) $750 - $1,000 0.030225 0.027727  0.014890 0.014890
Womens E-Mail     No E-Mail history_segment   2) $100 - $200 0.221022 0.226978 -0.014286 0.014286
  Mens E-Mail     No E-Mail        zip_code            Rural 0.152204 0.147329  0.013659 0.013659
  Mens E-Mail Womens E-Mail history_segment   4) $350 - $500 0.098418 0.102305 -0.012935 0.012935
  Mens E-Mail Womens E-Mail history_segment      7) $1,000 + 0.021777 0.020012  0.012338 0.012338
  Mens E-Mail     No

## 三、训练、验证、测试
只按实验分组分层；验证集选择模型和预算；测试集不参与选择。

In [4]:
from src.modeling import split_data
train, validation_ids, test = split_data(df)
print('样本量：', len(train), len(validation_ids), len(test))
print('训练与测试交集：', len(set(train) & set(test)))

样本量： 38400 12800 12800
训练与测试交集： 0
